# 注意力（Attention）

注意力不只是 Transformer 的一部分，注意力就是 Transformer。

看这句话：The cat sat on the mat because it was warm.
it 指什么？人知道它指的是 mat，但计算机怎么判断？

在注意力出现之前，模型只能从左到右逐个读词。
读到 it 时，mat 早已远去，信息已经衰减，模型会遗忘。

注意力让每个词能同时看到所有其他词。
读到 it 时，模型可以回看之前的词，判断哪个最重要——
这里会关注 mat，因为「温暖」是物体的属性，不是动物的。

注意力通过 query、key、value 决定对每个词的关注程度。
可以把它想象成在派对上：你的 query 是你想找什么；
其他人各有 key，表示他们能提供什么；匹配度高就多关注那个人。

本 notebook 从零实现完整的注意力机制，
并用真实数字走一遍示例，让你看清现代语言模型内部发生了什么。

## 本 notebook 的规模

我们按 GPT-2 Small 的规模构建注意力：
d_model=768，num_heads=12，每个 head 64 维。
完整注意力层约有 230 万参数。

小到能在笔记本上运行，又足够展示真实语言模型如何工作。
规模变大时数学不变，只是数字更大。

放大时架构不变，只是维度增长。以下数字来自 OpenAI 发表的 GPT-3 论文。

| 模型 | 层数 | d_model | 头数 | 参数量 |
|---|---|---|---|---|
| GPT-2 Small | 12 | 768 | 12 | 124M |
| GPT-2 Medium | 24 | 1024 | 16 | 350M |
| GPT-2 Large | 36 | 1280 | 20 | 774M |
| GPT-3 350M | 24 | 1024 | 16 | 350M |
| GPT-3 1.3B | 24 | 2048 | 24 | 1.3B |
| GPT-3 6.7B | 32 | 4096 | 32 | 6.7B |
| GPT-3 175B | 96 | 12288 | 96 | 175B |
| LLaMA 7B | 32 | 4096 | 32 | 7B |
| LLaMA 13B | 40 | 5120 | 40 | 13B |
| LLaMA 70B | 80 | 8192 | 64 | 70B |

GPT-3 175B 用的就是即将编写的同一套注意力公式，
只是 96 层而非 12 层，12288 维而非 768 维。
代码相同，数字大约大一千倍。

消费级 GPU 跑不动 GPT-3 175B，
需要多块数据中心 GPU 和巨额电费。
但本 notebook 在可玩、可理解的规模上抓住了同样的思想。

## 导入

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## 旋转位置嵌入（RoPE）

需要上一章的 RoPE：旋转 query 和 key 向量，
使注意力分数依赖相对位置。

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## 因果掩码（Causal Mask）

训练时每个 token 只能看到它之前的 token，
防止模型偷看未来词而作弊。

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## 多头注意力

这是完整的注意力模块，按顺序做八件事：
投影得到 Q、K、V；重塑为多个 head；应用 RoPE；
计算缩放点积分数；应用因果掩码；softmax；
对 value 加权；合并 head 并输出投影。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## 测试注意力模块

创建 12 个注意力 head 的模型，
输入 8 条长度为 64 的序列。

In [ ]:
attn = MultiHeadAttention(d_model=768, num_heads=12)

batch_size = 8
seq_len = 64
x = torch.randn(batch_size, seq_len, 768)
mask = create_causal_mask(seq_len, x.device)

output = attn(x, mask)
print(f"输入形状:  {x.shape}")
print(f"输出形状: {output.shape}")
print(f"输入输出形状一致: {x.shape == output.shape}")
print()

params = sum(p.numel() for p in attn.parameters())
print(f"注意力参数量: {params:,}")

## 用真实数字的完整示例

用仅 3 个 token 的短句走一遍注意力。
d_model=4、单 head，便于看清每个数字。

In [ ]:
tiny_attn = MultiHeadAttention(d_model=4, num_heads=1)

tokens = torch.tensor([[
    [0.5,  0.2, -0.3,  0.8],
    [0.1, -0.5,  0.7, -0.2],
    [0.9,  0.3, -0.1, -0.5],
]])
print(f"输入 token 形状: {tokens.shape}")
print()

with torch.no_grad():
    qkv = tiny_attn.qkv_proj(tokens)
    qkv = qkv.reshape(1, 3, 3, 1, 4)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

print("Query 向量:")
for i in range(3):
    vals = [f"{v:.3f}" for v in q[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

print("Key 向量:")
for i in range(3):
    vals = [f"{v:.3f}" for v in k[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

scores = (q @ k.transpose(-2, -1)) / math.sqrt(4)
print("注意力分数（掩码与 softmax 之前）:")
print(f"  Token 0: {[f'{v:.3f}' for v in scores[0, 0, 0].tolist()]}")
print(f"  Token 1: {[f'{v:.3f}' for v in scores[0, 0, 1].tolist()]}")
print(f"  Token 2: {[f'{v:.3f}' for v in scores[0, 0, 2].tolist()]}")

## 应用因果掩码与 softmax

因果掩码把未来位置设为负无穷，
softmax 后这些位置变为 0。
每个 token 只能关注自身及之前的 token。

In [ ]:
mask = create_causal_mask(3, scores.device)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("注意力权重（掩码与 softmax 之后）:")
for i in range(3):
    row = [f"{v:.3f}" for v in weights[0, 0, i].tolist()]
    print(f"  Token {i} 关注: [{', '.join(row)}]")
print()
print("Token 0 只看自己；Token 1 看 0 和 1；Token 2 看全部三个。")
print()

output = weights @ v
print("输出向量（value 的加权和）:")
for i in range(3):
    vals = [f"{v:.3f}" for v in output[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")

## 可视化因果注意力矩阵

每行是一个 token，每列是它关注的目标。
上三角为空，因为未来 token 被屏蔽。

In [ ]:
print("因果注意力矩阵（Token i 关注 Token j）:")
print("          T0      T1      T2")
for i in range(3):
    row = "  ".join([f"{weights[0, 0, i, j]:.3f}" for j in range(3)])
    print(f"T{i}       {row}")
print()
print("右上为零，模型无法看到未来。")